# Financial Fraud Detection — Autoresearch Results

**Dataset:** [Large-Scale Financial Fraud Dataset](https://www.kaggle.com/datasets/mdmahfuzsumon/large-scale-financial-fraud-dataset) (1M rows, 24 features)

**Task:** Binary classification — predict `is_fraud` from transaction attributes

**Approach:** Autonomous ML experiment loop ([autoresearch](https://github.com/detrin/autoresearch)) — 13 experiments run by an AI agent, logged to MLflow

**Best Result:** AUC-ROC **0.8845** (5-model stacking ensemble)

| # | Model | Description | AUC-ROC | Improvement |
|---|-------|-------------|---------|-------------|
| 1 | LogisticRegression | Baseline, label-encoded | 0.8663 | — |
| 2 | RandomForest | 200 trees | 0.8835 | +1.7% |
| 3 | LGBMClassifier | 500 trees | 0.8834 | +1.7% |
| 4 | LGBMClassifier | 1000 trees, scale_pos_weight | 0.8836 | +1.7% |
| 5 | LGBMClassifier | + feature eng (amount/fee ratio) | 0.8828 | worse |
| 6 | LGBMClassifier | native categorical handling | 0.8825 | worse |
| 7 | LGBMClassifier | early stop + regularization | 0.8824 | worse |
| 8 | LGBMClassifier | auc eval + early stop 100 | 0.8838 | +1.8% |
| 9 | RandomForest | 500 trees, balanced weights | 0.8826 | worse |
| 10 | LGBMClassifier | + target encoding city/country | 0.8834 | no change |
| 11 | Ensemble (2xLGBM+RF) | weighted average | 0.8842 | +1.8% |
| 12 | Ensemble (3xLGBM+RF+ET) | 5-model stacking | **0.8845** | **+1.8%** |
| 13 | Ensemble (4xLGBM+RF+ET) | 6-model stacking | 0.8844 | no gain |

**Key Insight:** Tree-based models all converged to ~0.884 AUC. Feature engineering, hyperparameter tuning, and ensembling provided marginal gains. The dataset likely has a hard ceiling due to synthetic generation.

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.preprocessing import LabelEncoder

DATA_FILE = "improved_fraud_dataset.csv"
kaggle_matches = glob.glob(f"/kaggle/input/**/{DATA_FILE}", recursive=True)
if kaggle_matches:
    DATA_PATH = kaggle_matches[0]
elif os.path.exists(DATA_FILE):
    DATA_PATH = DATA_FILE
else:
    raise FileNotFoundError(f"Cannot find {DATA_FILE}")

print(f"Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(f"Fraud rate: {df['is_fraud'].mean():.4f} ({df['is_fraud'].sum():,} / {len(df):,})")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")
print(f"\nNull values: {df.isnull().sum().sum()}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['is_fraud'].value_counts().plot.bar(ax=axes[0, 0], edgecolor='black')
axes[0, 0].set_title('Fraud Distribution')
axes[0, 0].set_xticklabels(['Not Fraud', 'Fraud'], rotation=0)

df.groupby('merchant_category')['is_fraud'].mean().sort_values().plot.barh(ax=axes[0, 1])
axes[0, 1].set_title('Fraud Rate by Merchant Category')

axes[1, 0].hist(df[df['is_fraud']==0]['transaction_amount'], bins=50, alpha=0.5, label='Not Fraud', density=True)
axes[1, 0].hist(df[df['is_fraud']==1]['transaction_amount'], bins=50, alpha=0.5, label='Fraud', density=True)
axes[1, 0].set_title('Transaction Amount Distribution')
axes[1, 0].legend()

df.groupby('hour')['is_fraud'].mean().plot(ax=axes[1, 1])
axes[1, 1].set_title('Fraud Rate by Hour')
axes[1, 1].set_xlabel('Hour')

plt.tight_layout()
plt.show()

## 3. Data Preparation

In [ ]:
TARGET = "is_fraud"
RANDOM_STATE = 42

train, val = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[TARGET])

drop_cols = [TARGET, "transaction_id", "user_id", "organization", "transaction_timestamp"]
feature_cols = [c for c in train.columns if c not in drop_cols]

cat_cols = train[feature_cols].select_dtypes(include="object").columns.tolist()
print(f"Categorical: {cat_cols}")
print(f"Numeric: {[c for c in feature_cols if c not in cat_cols]}")
print(f"Train: {train.shape}, Val: {val.shape}")

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    val[col] = le.transform(val[col].astype(str))
    encoders[col] = le

X_train, y_train = train[feature_cols], train[TARGET]
X_val, y_val = val[feature_cols], val[TARGET]

## 4. Baseline — Logistic Regression

AUC-ROC: **0.8663**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
probs_lr = lr.predict_proba(X_val)[:, 1]
auc_lr = roc_auc_score(y_val, probs_lr)
print(f"Logistic Regression AUC-ROC: {auc_lr:.4f}")

results = [("LogisticRegression", auc_lr)]

## 5. Random Forest

AUC-ROC: **0.8835** (+1.7%)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
probs_rf = rf.predict_proba(X_val)[:, 1]
auc_rf = roc_auc_score(y_val, probs_rf)
print(f"RandomForest AUC-ROC: {auc_rf:.4f}")

results.append(("RandomForest", auc_rf))

## 6. Best Model — LightGBM with Early Stopping

The stacking ensemble (AUC 0.8845) ran out of memory on Kaggle's 16GB limit. The single best LightGBM variant achieved nearly identical performance at AUC 0.8838 — demonstrating that the ensemble gain was marginal.

AUC-ROC: **0.8838**

In [ ]:
import lightgbm as lgb

best_model = lgb.LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
)
probs_best = best_model.predict_proba(X_val)[:, 1]
auc_best = roc_auc_score(y_val, probs_best)
print(f"LightGBM AUC-ROC: {auc_best:.4f}")

results.append(("LightGBM (best)", auc_best))

## 7. ROC Curve & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, probs in [("Logistic", probs_lr), ("RandomForest", probs_rf), ("LightGBM", probs_best)]:
    fpr, tpr, _ = roc_curve(y_val, probs)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val, probs):.4f})")
axes[0].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

importance = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values()
importance.tail(15).plot.barh(ax=axes[1])
axes[1].set_title('Top 15 Features (LightGBM split importance)')

plt.tight_layout()
plt.show()

## 8. Results Summary

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "AUC-ROC"]).sort_values("AUC-ROC", ascending=False)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ecc71" if m == "Stacking Ensemble" else "#3498db" for m in results_df["Model"]]
bars = ax.barh(results_df["Model"], results_df["AUC-ROC"], color=colors, edgecolor="black")
ax.set_xlabel("AUC-ROC (higher is better)")
ax.set_title("Model Comparison — Financial Fraud Detection")
ax.set_xlim(0.85, 0.89)
for bar, val in zip(bars, results_df["AUC-ROC"]):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2, f"{val:.4f}", va="center")
plt.tight_layout()
plt.show()

## Conclusions

**Best model:** 5-model stacking ensemble — **AUC-ROC 0.8845** (+2.1% over baseline)

**What worked:**
- Tree-based models immediately jumped from 0.866 to 0.883 — the single biggest gain
- Stacking diverse models (3 LightGBM variants + RandomForest + ExtraTrees) squeezed out the last bit

**What didn't work:**
- Feature engineering (amount/fee ratios, frequency features) — no gain
- Target encoding categorical variables — no improvement over label encoding
- Heavier regularization / early stopping — slightly worse
- Adding more models to the ensemble beyond 5 — no further gain

**Takeaway:** All tree-based approaches converged to ~0.884 AUC-ROC. After 13 experiments, the improvement space was exhausted. The dataset appears to have a hard ceiling — likely due to synthetic data generation patterns. For a real fraud detection system, the focus should be on threshold optimization (precision/recall tradeoff) rather than pushing AUC higher.

---
*Generated via [autoresearch](https://github.com/detrin/autoresearch) — 13 autonomous experiments, MLflow-tracked*